# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, duckdb, os
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs_v2"

data_model = pd.read_csv(f"{BASE}/data_model_v2.csv")
print("Loaded:", data_model.shape)
print(data_model.columns.tolist())

Mounted at /content/drive
Loaded: (183345, 23)
['client_hash_id', 'content_hash_id', 'impressions_window', 'clicks_window', 'april_impressions', 'april_clicks', 'february_clicks', 'click_through_rate', 'weighted_position', 'momentum', 'active_days', 'click_through_rate_missing', 'weighted_position_missing', 'momentum_missing', 'clicks_april', 'clicks_may', 'declined', 'momentum_risk', 'baseline_score', 'reason_code', 'action', 'content_age_days', 'content_age_days_missing']


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Finding A—"What Predicts Health?"(Random Forest feature importance, Position 43%, Impressions 32%)

My methodology question: Where does the label (Health Score) come from? Per the paper's own methodology, Health Score is built directly from Impressions, Position, CTR, and Scroll Depth. So the model's top predictors are largely restating the label's own ingredients back as "findings" — the leakage taxonomy's first pattern (label-derived features). The paper discloses this honestly, which is the right move — my constructive question is whether a train-without-Position/Impressions test would show the expected collapse, confirming this is descriptive rather than predictive.

# Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

My methodology questions: What's the base rate (share of pages actually growing vs. declining in the 61.8K sample)? Without it, 71% can't be judged as skill or near-majority-class guessing. Was the 80/20 split grouped by brand (57 brands present) or random? A random split risks the exact client/brand-memorization effect I found in my own Week 5 model (see Section 2) — a brand-grouped split would answer the more honest, deployment-relevant question.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

model_cols = ["impressions_window", "clicks_window", "april_impressions", "april_clicks",
              "february_clicks", "click_through_rate", "weighted_position", "momentum",
              "active_days", "click_through_rate_missing", "weighted_position_missing", "momentum_missing"]

X = data_model[model_cols]
y = data_model["declined"]
groups = data_model["client_hash_id"]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    y_arr = y_true.values if hasattr(y_true, "values") else np.asarray(y_true)
    return y_arr[order[:k]].mean()

# --- BEFORE: naive random split ---
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.3, random_state=42)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]

y_test_r_reset = y_test_r.reset_index(drop=True)
auc_random = roc_auc_score(y_test_r_reset, scores_random)
p50_random = precision_at_k(y_test_r_reset, scores_random, 50)

overlap_random = len(set(data_model.iloc[X_train_r.index]["client_hash_id"]) &
                      set(data_model.iloc[X_test_r.index]["client_hash_id"]))

print("RANDOM SPLIT: AUC =", auc_random, "| Precision@50 =", p50_random, "| Client overlap =", overlap_random)

RANDOM SPLIT: AUC = 0.9276913585369413 | Precision@50 = 1.0 | Client overlap = 49


In [9]:
# --- AFTER: grouped split by client ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_grouped.fit(X_train_g, y_train_g)
scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

y_test_g_reset = y_test_g.reset_index(drop=True)
auc_grouped = roc_auc_score(y_test_g_reset, scores_grouped)
p50_grouped = precision_at_k(y_test_g_reset, scores_grouped, 50)

overlap_grouped = len(set(data_model.iloc[train_idx]["client_hash_id"]) & set(data_model.iloc[test_idx]["client_hash_id"]))

print("GROUPED SPLIT: AUC =", auc_grouped, "| Precision@50 =", p50_grouped, "| Client overlap =", overlap_grouped)

GROUPED SPLIT: AUC = 0.9302290957526194 | Precision@50 = 0.54 | Client overlap = 0


In [10]:
before_after = pd.DataFrame({
    "split": ["Random (naive)", "Grouped by client (honest)"],
    "AUC": [auc_random, auc_grouped],
    "precision@50": [p50_random, p50_grouped],
    "client_overlap": [overlap_random, overlap_grouped]
})
before_after

,split,AUC,precision@50,client_overlap
0,Random (naive),0.927691,1.00,49
1,Grouped by client (honest),0.930229,0.54,0


Before/after, the honest way: a naive random split showed a perfect precision@50 (100%) with AUC of 0.928. Checking client overlap revealed 49 clients present in both train and test — meaning the model could partially recognize specific clients rather than learning a pattern that generalizes. Re-running under a client-grouped split (zero overlap) dropped precision@50 to a still-strong but much more honest 54%, while AUC barely moved (0.930). The gap between 100% and 54% is the actual finding: nearly half of the apparent top-of-queue perfection in the naive split was a memorization artifact, not real predictive skill. The grouped-split number (54%) is the one that should be trusted and reported — it reflects genuine performance on clients the model has never seen any pages from, which is the honest, deployment-relevant question.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [11]:
print("--- Attack Checklist ---\n")

# 1. Timeline
print("1. Timeline: features from Feb-Apr only. Label (declined) from clicks_april vs clicks_may.")
print("   april_clicks is a feature AND shares its reference month with the label's baseline —")
print("   flagged, tested below.\n")

# 2. Train-without test on the suspect
model_cols_no_april = [c for c in model_cols if c not in ["april_clicks", "clicks_window"]]
X_no_april = data_model[model_cols_no_april]
X_train_na, X_test_na = X_no_april.iloc[train_idx], X_no_april.iloc[test_idx]

rf_no_april = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf_no_april.fit(X_train_na, y_train_g)
scores_no_april = rf_no_april.predict_proba(X_test_na)[:, 1]
auc_no_april = roc_auc_score(y_test_g_reset, scores_no_april)

print(f"2. AUC WITH april_clicks: {auc_grouped:.4f}")
print(f"   AUC WITHOUT april_clicks: {auc_no_april:.4f}")
collapse = auc_grouped - auc_no_april
print(f"   Collapse: {collapse:.4f} —", "MINOR, likely real signal" if collapse < 0.1 else "LARGE, investigate")

# 3. Product flags
excluded_flags = ["is_deleted", "is_published", "health_score", "optimization_flags"]
present = [c for c in excluded_flags if c in model_cols]
print(f"\n3. Product/decision flags present: {present} (must be empty)")

# 4. Split grouping
print(f"\n4. Split grouped by client_hash_id: YES, overlap = {overlap_grouped} (must be 0)")

# 5. Base rate
print(f"\n5. Base rate (test set): {y_test_g_reset.mean():.4f}")

# 6. Feature importance sanity check
importances = pd.Series(rf_grouped.feature_importances_, index=model_cols).sort_values(ascending=False)
print(f"\n6. Feature importances:\n{importances}")
print(f"   Top feature share: {importances.iloc[0]:.1%} —",
      "investigate" if importances.iloc[0] > 0.7 else "no single feature dominates")

--- Attack Checklist ---

1. Timeline: features from Feb-Apr only. Label (declined) from clicks_april vs clicks_may.
   april_clicks is a feature AND shares its reference month with the label's baseline —
   flagged, tested below.

2. AUC WITH april_clicks: 0.9302
   AUC WITHOUT april_clicks: 0.9041
   Collapse: 0.0262 — MINOR, likely real signal

3. Product/decision flags present: [] (must be empty)

4. Split grouped by client_hash_id: YES, overlap = 0 (must be 0)

5. Base rate (test set): 0.1888

6. Feature importances:
april_clicks                  0.419269
click_through_rate            0.181487
clicks_window                 0.120373
weighted_position_missing     0.119007
momentum                      0.065228
april_impressions             0.042163
impressions_window            0.024558
february_clicks               0.008913
weighted_position             0.007670
active_days                   0.006805
momentum_missing              0.004528
click_through_rate_missing    0.000000
dtyp

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest original claim: "Random Forest achieves 0.930 AUC, decisively outperforming the baseline."

What the audit found: this claim is only half-true and needs correcting on the specific metric that matters for this lane. Random Forest's AUC (0.930) is genuinely strong and observed to exceed the volume baseline's AUC (0.786) — but at precision@50, the metric matching actual weekly review capacity, the simple volume-based baseline (56%) actually exceeds Random Forest (54%).

Rewritten in safe language:
"On a client-grouped holdout, Random Forest shows an observed AUC of 0.930, indicating stronger overall ranking quality across the full portfolio than the volume-only baseline (AUC 0.786). However, at precision@50 — the measured, decision-relevant metric for a 50-page weekly review queue — the baseline (56%) narrowly exceeds Random Forest (54%). This is a directional finding, not a universal claim that either method is 'better': Random Forest generalizes more broadly, while the baseline's simplicity is unexpectedly robust at the specific top-of-queue size that matters for decision-support use.

Error analysis suggests Random Forest's few highest-confidence mistakes cluster in one client's largest pages — a concrete, disclosed limitation rather than a hidden one."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.